In [2]:
from conv import *
from scState_model import *
from utils import *
import anndata as ad
from collections import Counter
import copy
import dill
from functools import partial
import json
import math
import matplotlib.cm as cm
import matplotlib.pyplot as plt
import multiprocessing as mp
import numpy as np
import os
import pandas as pd
from operator import itemgetter
import random
import scipy.sparse as sp
from scipy.io import mmread
from scipy.sparse import hstack, vstack, coo_matrix
import seaborn as sb
from sklearn import metrics
from sklearn.cluster import KMeans
from sklearn.decomposition import IncrementalPCA
from sklearn.decomposition import SparsePCA
from sklearn.metrics import accuracy_score
from sklearn.metrics.cluster import normalized_mutual_info_score
import time
import torch
import torch.cuda as cuda
from torch import nn
from torch.autograd import Variable
import torch.distributions as D
import torch.nn.functional as F
import torch_geometric.data as Data
from torch_geometric.nn import GCNConv, GATConv
from torch_geometric.nn.conv import MessagePassing
from torch_geometric.nn.inits import glorot, uniform
from torch_geometric.utils import softmax as Softmax
from torchmetrics.functional import pairwise_cosine_similarity
import warnings
from warnings import filterwarnings
import xlwt
import argparse
from tqdm import tqdm
import scanpy as sc
from scipy import sparse
from scipy.sparse import csc_matrix,csr_matrix

dataset_name = 'GSE147298_McSC'
batch_remove = False

filterwarnings("ignore")

parser = argparse.ArgumentParser(description='Training GNN on gene cell graph')
parser.add_argument('--seed', type=int, default=0)
parser.add_argument('--fi', type=int, default=0) # This parameter is used for the benchmark to specify the starting sequence number of the created files
parser.add_argument('--labsm', type=float, default=0.1) # The rate of LabelSmoothing
parser.add_argument('--wd', type=float, default=0.1) # The 'weight_decay' parameter is used to specify the strength of L2 regularization
parser.add_argument('--lr', type=float, default=0.0005) # learning rate 0.0005
parser.add_argument('--n_hid', type=int, default=104) # The number of layers should be a multiple of 'n_head' in order to make any modifications
parser.add_argument('--nheads', type=int, default=8) # The 'heads' parameter represents the number of attention heads in the attention mechanism
parser.add_argument('--nlayers', type=int, default=3) # The number of layers in network
parser.add_argument('--cell_size', type=int, default=30) # The number of cells per subgraph (batch)
parser.add_argument('--neighbor', type=int, default=20) # The number of neighboring nodes to be selected for each cell in the subgraph
parser.add_argument('--egrn', type=bool, default=False) # Whether to output the Enhancer-Gene regulatory network
parser.add_argument('--epochs', type=int, default=3) # The epoch number of NodeDimensionReduction
parser.add_argument('--num_epochs', type=int, default=3) # The epoch number of scState-Model
parser.add_argument('--output_file', type=str, default= f'Result/') # Please choose an output path to replace this path on your own.
args = parser.parse_args([])

output_file = args.output_file
fi=args.fi
labsm = args.labsm
lr = args.lr
wd = args.wd
n_hid = args.n_hid
nheads = args.nheads
nlayers = args.nlayers
cell_size = args.cell_size
neighbor = args.neighbor
egrn = args.egrn
epochs = args.epochs
num_epochs = args.num_epochs

seed = args.seed
random.seed(seed)
np.random.seed(seed)
torch.manual_seed(seed)
torch.cuda.manual_seed(seed)
torch.cuda.manual_seed_all(seed)
os.environ['PYTHONHASHSEED'] = str(seed)
torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False

In [5]:
# 读取h5ad数据作为输入
adata = sc.read_h5ad(f"Data/{dataset_name}_adata.h5ad")

In [6]:
import numpy as np
from scipy.sparse import csr_matrix

gene_names = adata.var_names.to_numpy()
cell_names = adata.obs_names.to_numpy()

X = adata.X

if isinstance(X, np.ndarray):
    RNA_matrix = csr_matrix(X.T)
else:
    RNA_matrix = X.T.tocsr()

GSVA = adata.obsm["GSVA"]
GSVA_matrix = GSVA.T
GSVA_matrix = csr_matrix(GSVA_matrix)

gepa_names = np.array(adata.uns["GSVA_pathways"])

In [7]:
# ============================================ 预聚类 ====================================================
device = torch.device("cuda" if cuda.is_available() else "cpu")
print('You will use : ',device)
# clustering result by scanpy
if batch_remove:
    matrix = adata.obsm["RBE"]
else:
    matrix = RNA_matrix
initial_pre = initial_clustering(matrix, batch_remove=batch_remove) 
# number of every cluster
cluster_ini_num = len(set(initial_pre)) 
print(f'init cluster number is {cluster_ini_num}')
ini_p1 = [int(i) for i in initial_pre] 
# partite the data into batches
indices, Node_Ids, dic = batch_select_whole(RNA_matrix, GSVA_matrix, neighbor = [neighbor], cell_size=cell_size)
n_batch = len(indices)

You will use :  cpu
	Using original count matrix for pre-clustering.
         Falling back to preprocessing with `sc.pp.pca` and default params.
init cluster number is 6
We are currently in the process of partitioning the data into batches. Kindly wait for a moment, please.


100%|██████████| 60/60 [00:53<00:00,  1.13it/s]


In [8]:
adata = run_palantir_pipeline(adata)

Determing nearest neighbor graph...
Sampling and flocking waypoints...
Time for determining waypoints: 0.0033515493075052896 minutes
Determining pseudotime...
Shortest path distances using 30-nearest neighbor graph...


findfont: Font family ['Raleway'] not found. Falling back to DejaVu Sans.
findfont: Font family ['Raleway'] not found. Falling back to DejaVu Sans.
findfont: Font family ['Raleway'] not found. Falling back to DejaVu Sans.
findfont: Font family ['Raleway'] not found. Falling back to DejaVu Sans.
findfont: Font family ['Raleway'] not found. Falling back to DejaVu Sans.
findfont: Font family ['Raleway'] not found. Falling back to DejaVu Sans.
findfont: Font family ['Raleway'] not found. Falling back to DejaVu Sans.
findfont: Font family ['Raleway'] not found. Falling back to DejaVu Sans.
findfont: Font family ['Raleway'] not found. Falling back to DejaVu Sans.
findfont: Font family ['Raleway'] not found. Falling back to DejaVu Sans.
findfont: Font family ['Raleway'] not found. Falling back to DejaVu Sans.
findfont: Font family ['Raleway'] not found. Falling back to DejaVu Sans.
findfont: Font family ['Raleway'] not found. Falling back to DejaVu Sans.
findfont: Font family ['Raleway'] not 

Time for shortest paths: 0.20418750445048015 minutes
Iteratively refining the pseudotime...
Correlation at iteration 1: 0.9996
Correlation at iteration 2: 1.0000
Entropy and branch probabilities...
Markov chain construction...
Identification of terminal states...
Computing fundamental matrix and absorption probabilities...
Project results to all cells...


In [10]:
# Model node reduction training
node_model = NodeDimensionReduction(RNA_matrix, GSVA_matrix, indices, ini_p1, n_hid=n_hid, n_heads=nheads, 
                                    n_layers=nlayers,labsm=labsm, lr=lr, wd=wd, device=device, num_types=3, num_relations=2, epochs=100)
gnn, cell_embedding, cell_clu = node_model.train_model(n_batch=n_batch)

# Model training stage1
scState_stage1_model = scState_stage1(gnn=gnn, labsm=labsm, n_hid=n_hid, n_batch=n_batch, device=device,lr=lr,palantir_pseudotime=np.array(adata.obs['palantir_pseudotime']), Node_Ids=Node_Ids, wd=wd, num_epochs=50)
scState_gnn_stage1, is_stem_mask, cell_embedding, stem_like_clus = scState_stage1_model.train_model(indices=indices,RNA_matrix=RNA_matrix, GSVA_matrix=GSVA_matrix, ini_p1=ini_p1)

# Model training stage2
prototype_indicies, labeled_target_data, stem_labels_torch, gmm_ratio = stage2_initial_clustering(GSVA_matrix, Node_Ids, is_stem_mask, device)
scState_stage2_model = scState_stage2(gnn=scState_gnn_stage1, labsm=labsm, n_hid=n_hid, n_batch=n_batch, device=device,lr=lr,Node_Ids=Node_Ids, wd=wd, cell_embedding=cell_embedding, prototype_indicies=prototype_indicies, is_stem_mask=is_stem_mask, stem_pseudo_labels=stem_labels_torch, labeled_target_data=labeled_target_data, gmm_ratio=gmm_ratio, indices=indices,RNA_matrix=RNA_matrix, GSVA_matrix=GSVA_matrix, num_epochs=150)
scState_gnn_stage2, prototypes = scState_stage2_model.train_model(indices=indices,RNA_matrix=RNA_matrix, GSVA_matrix=GSVA_matrix, ini_p1=ini_p1)

# The result of scState
scState_result = scState_pred_stage2(RNA_matrix, GSVA_matrix, egrn=egrn, scState_gnn=scState_gnn_stage2, indices=indices, 
                    nodes_id=Node_Ids, cell_size=cell_size, device=device, gene_names=gene_names, gepa_names=gepa_names, stem_clu=stem_like_clus, prototypes=prototypes)

nodes_id = list(Node_Ids)
pred_label = list(scState_result['pred_label'])
cell_embedding = scState_result['cell_embedding']

# Save numpy arrays to files
np.save(output_file + f"/Node_Ids.npy", Node_Ids)
np.save(output_file + f"/pred.npy", scState_result['pred_label'])
np.save(output_file + f"/cell_embedding.npy", scState_result['cell_embedding'])

The training process for the NodeDimensionReduction model has started. Please wait.


100%|██████████| 3/3 [01:13<00:00, 24.42s/it]


The training for the NodeDimensionReduction model has been completed.
The training process for the scState model has started. Please wait.
The 0 epoch


100%|██████████| 60/60 [00:12<00:00,  4.90it/s]


The 1 epoch


100%|██████████| 60/60 [00:11<00:00,  5.18it/s]


The training for the scState model has been completed.
The training process for the scState model has started. Please wait.


100%|██████████| 3/3 [01:31<00:00, 30.44s/it]


The training for the scState model has been completed.
